<a href="https://colab.research.google.com/github/MuhammadRhakan/final_project/blob/main/3_Refined_Methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk
import kagglehub

import pandas as pd
import numpy as np
import networkx as nx

from kagglehub.datasets import KaggleDatasetAdapter
from sklearn.preprocessing import normalize, StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.metrics import pairwise_distances
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import *
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix


nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
course = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "hossaingh/udemy-courses", "Course_info.csv")

<ipython-input-3-193580186>:1: DeprecationWarning: load_dataset is deprecated and will be removed in a future version.
  course = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "hossaingh/udemy-courses", "Course_info.csv")


In [ ]:
course = course[course['language'].isin({'English'})]

In [ ]:
def missing_values(course):
  null_values = course[(course['num_lectures'] == 0) |
                (course['content_length_min'] == 0) |
                ((course['avg_rating'] == 0) & (course['num_reviews'] > 0))].index
  clean_data = course.drop(index=null_values).dropna()

  return clean_data

In [ ]:
course = missing_values(course)

In [ ]:
#separate numeric and non-numeric attributes
def attributes(data, shift='avg_rating'):
  categorical = []
  numerical = []

  for i, cat in enumerate(data.select_dtypes(include = ['object', 'bool']).columns.values):
    categorical.append(cat)
  categorical.append(shift)

  for i, num in enumerate(data.select_dtypes(include = 'number').drop(columns='id').columns.values):
    if num != shift:
      numerical.append(num)

  return categorical, numerical


#prepare clean data
def data_cleaning(data, features, par=0.9):
  outliers_indices = set()
  transformer = PowerTransformer(method='box-cox')

  for col in features:
    exclude = data[col].quantile(par)
    outliers = data[data[col] > exclude]
    outliers_indices.update(outliers.index)

  trim = data.drop(index=outliers_indices)
  transformed = trim.copy()

  transformed[['num_subscribers', 'num_reviews', 'num_comments']] = np.log1p(transformed[['num_subscribers', 'num_reviews', 'num_comments']])
  transformed[['price', 'num_lectures', 'content_length_min']] = np.sqrt(transformed[['price', 'num_lectures', 'content_length_min']])

  return trim, transformed


#modularize each feature type
def features_type(data):
  return {
      'semantic': ['title', 'headline'],
      'nominal': ['is_paid', 'category', 'subcategory'],
      'datetime': ['published_time', 'last_update_date'],
      'high_cardinal': 'instructor_name',
      'ordinal': 'avg_rating'}


#feature engineering for high-cardinality categorical data
def calc_smoothed_instructor_rating(data, feature, rating='avg_rating', subscriber='num_subscribers', weight=50):
  data['engagement'] = data[rating] * data[subscriber]

  instructor_stats = data.groupby(feature).agg(
      total_rating=('engagement', 'sum'),
      total_subs=(subscriber, 'sum'))

  instructor_stats['weighted_avg'] = instructor_stats.apply(lambda row: row['total_rating'] / row['total_subs'] if row['total_subs'] > 0 else 0, axis=1)
  # instructor_stats['weighted_avg'] = instructor_stats['total_rating'] / instructor_stats['total_subs']
  global_avg = data['engagement'].sum() / data[subscriber].sum()
  instructor_stats['smoothed'] = (
      (instructor_stats['total_subs'] * instructor_stats['weighted_avg'] + weight * global_avg) /
      (instructor_stats['total_subs'] + weight))

  data['instructor_score'] = data[feature].map(instructor_stats['smoothed'])
  data[['avg_rating', 'instructor_score']] = data[['avg_rating', 'instructor_score']].astype('int64')

  return data[['avg_rating', 'instructor_score']]


#label ordinal features to categories (nominal)
def ordinal_feature_reengineering(data):
  data = data.copy()
  for col in data.columns:
      data.loc[:, col] = data[col].apply(lambda x: 'Low' if x < 3 else ('Medium' if x == 3 else 'High'))
  return data


#combine transformed features with nominal features
def combined_features(data_1, data_2):
  return pd.concat([data_1, data_2], axis=1)

In [ ]:
categorical, numerical = attributes(course)
course_clean, course_clean_scaled = data_cleaning(course, numerical)
types = features_type(course)
ordinal_mod = calc_smoothed_instructor_rating(course_clean_scaled, types['high_cardinal'])
rating = ordinal_feature_reengineering(ordinal_mod)
types['nominal'] = combined_features(course_clean_scaled[types['nominal']], rating)

In [ ]:
types['nominal']

,is_paid,category,subcategory,avg_rating,instructor_score
2,True,Lifestyle,Other Lifestyle,High,High
18,True,Design,Design Tools,High,High
19,True,Teaching & Academics,Science,Low,Medium
23,True,Teaching & Academics,Teacher Training,Medium,High
25,True,Health & Fitness,General Health,High,High
...,...,...,...,...,...
209724,True,Teaching & Academics,Other Teaching & Academics,Low,High
209725,True,Lifestyle,Esoteric Practices,High,High
209727,True,Teaching & Academics,Language Learning,Low,High
209729,True,Teaching & Academics,Language Learning,Low,Medium


In [ ]:
def semantic_preprocessing_batch(batch, features, stop_words, lemmatizer):
    batch = batch[features].copy()
    for col in features:
        batch[col] = batch[col].str.lower()

    for col in features:
        batch[col] = batch[col].apply(lambda text: ' '.join([lemmatizer.lemmatize(word) for word in word_tokenize(text) if word not in stop_words]))

    return batch.apply(lambda row: ' '.join(row), axis=1)


def semantic_preprocessing(data, features, batch_size=5000):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    vectorizer = TfidfVectorizer(max_features=1000, min_df=10, max_df=0.9, ngram_range=(1, 2), dtype=np.float32)

    combined_text = []
    for i in range(0, len(data), batch_size):
      batch = data.iloc[i:i+batch_size]
      processed = semantic_preprocessing_batch(batch, features, stop_words, lemmatizer)
      combined_text.extend(processed)

    tfidf_matrix = vectorizer.fit_transform(combined_text)
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

    similarity_batches = []
    for i in range(0, tfidf_matrix.shape[0], batch_size):
        batch = tfidf_matrix[i:i+batch_size]
        similarity_batch = cosine_similarity(batch, tfidf_matrix)
        similarity_batches.append(similarity_batch.astype(np.float16))

    return tfidf_matrix, np.vstack(similarity_batches)

In [ ]:
semantic_result, semantic_similarities = semantic_preprocessing(course_clean_scaled, types['semantic'])

In [ ]:
print(f'TF-IDF Matrix:  {semantic_result.shape[0]} rows x {semantic_result.shape[1]} words')
print(f'TF-IDF Similarity Matrix:   {semantic_similarities.shape[0]} items x {semantic_similarities.shape[1]} items')

TF-IDF Result:  85480 rows x 1000 words
Final Matrix:   85480 items x 85480 items


In [ ]:
def numerical_preprocessing(data, features, batch_size=5000, threshold=0.7, max_print=None):
  normalized_matrix = normalize(data[features])
  normalized_df = pd.DataFrame(data=normalized_matrix, columns=data[features].columns)

  similarity_batches = []

  for i in range(0, len(data), batch_size):
    batch = normalized_matrix[i:i+batch_size]
    similarity_batch = cosine_similarity(batch, normalized_matrix)
    similarity_batches.append(similarity_batch.astype(np.float16))
    similarity_matrix = np.vstack(similarity_batches)

    low_sim_indices = np.where((similarity_matrix < threshold) & (similarity_matrix < 0.999))

    # Prepare and optionally print results
    results = []
    for idx, (i, j) in enumerate(zip(*low_sim_indices)):
        results.append((i, j, similarity_matrix[i, j]))
        if max_print is not None and idx + 1 >= max_print:
            break

    for i, j, sim in results:
        print(f"Item {i} vs Item {j} = Similarity: {sim:.4f}")

  return normalized_matrix, np.vstack(similarity_batches), normalized_df

In [ ]:
numeric_result, numeric_similarities, normalized_data = numerical_preprocessing(course_clean_scaled, numerical)

In [ ]:
print(f'Numeric Matrix:  {numeric_result.shape[0]} rows x {numeric_result.shape[1]} features')
print(f'Numeric Similarity Matrix:   {numeric_similarities.shape[0]} items x {numeric_similarities.shape[1]} items')